# Build Sparse Title Index

This notebook reads the Wikipedia titles file from the local `retrieval/` directory and builds a PyTerrier sparse index in the same area.

In [1]:
from pathlib import Path

import pandas as pd
import pyterrier as pt

In [2]:
CWD = Path.cwd().resolve()
RETRIEVAL_DIR = CWD if (CWD / "build_sparse_title_index.ipynb").exists() else CWD / "retrieval"
INPUT_TITLES_FILE = RETRIEVAL_DIR / "enwiki-20250220-all-titles"
INDEX_OUTPUT_DIR = RETRIEVAL_DIR / "2025_sparse_wikititle_index"

print(f"Input file: {INPUT_TITLES_FILE}")
print(f"Index output dir: {INDEX_OUTPUT_DIR}")

Input file: E:\Intent_Slot_Code\retrieval\enwiki-20250220-all-titles
Index output dir: E:\Intent_Slot_Code\retrieval\2025_sparse_wikititle_index


In [3]:
def load_titles_dataframe(titles_file: Path) -> pd.DataFrame:
    if not titles_file.exists():
        raise FileNotFoundError(f"Titles file not found: {titles_file}")

    titles = []
    with titles_file.open("r", encoding="utf-8") as file:
        next(file)
        for line in file:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t", maxsplit=1)
            if len(parts) < 2:
                continue
            titles.append(parts[1].replace("_", " "))

    doc_ids = [f"doc{i + 1:08d}" for i in range(len(titles))]
    return pd.DataFrame({"docno": doc_ids, "title": titles})


wiki_titles_df = load_titles_dataframe(INPUT_TITLES_FILE)
wiki_titles_df.head()

,docno,title
0,doc00000001,!
1,doc00000002,!!
2,doc00000003,!!!
3,doc00000004,!!!!!!!
4,doc00000005,!!!Fuck You!!!


In [4]:
def build_sparse_index(df: pd.DataFrame, index_dir: Path):
    if not pt.started():
        pt.init(mem=100000)

    index_dir.mkdir(parents=True, exist_ok=True)
    properties = {
        "index.meta.data-source": "fileinmem",
        "index.meta.index-source": "fileinmem"
    }

    indexer = pt.IterDictIndexer(
        str(index_dir),
        verbose=True,
        meta={"docno": 20, "title": 256},
        text_attrs=["title"],
        properties=properties,
    )
    indexer.index(df.fillna("").to_dict(orient="records"))

    props_path = index_dir / "data.properties"
    index = pt.IndexFactory.of(str(props_path))
    print(index.getCollectionStatistics().toString())
    return index


In [5]:
index = build_sparse_index(wiki_titles_df, INDEX_OUTPUT_DIR)
print(f"Sparse index available at: {INDEX_OUTPUT_DIR}")

C:\Users\thiag\AppData\Local\Temp\ipykernel_97444\4115552778.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
C:\Users\thiag\AppData\Local\Temp\ipykernel_97444\4115552778.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
The following code will have the same effect:
pt.java.set_memory_limit(100000)
pt.java.init() # optional, forces java initialisation
  pt.init(mem=100000)


20:19:15.443 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (doc00000001) - further warnings are suppressed
21:28:40.287 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 38171 empty documents
22:38:54.079 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 234004 empty documents
23:50:00.570 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 563016 empty documents
00:23:44.751 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 575930 empty documents
00:23:46.826 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading lookup file directly from disk (SLOW) - try index.meta.index-source=fileinmem in the index properties file. 137,3 MiB of memory would be required.
00:23:46.856 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 9